# Lab 3 demo: churn model + RAG retention offers
End-to-end walkthrough: evaluation evidence, test-split scoring, top-10% flagged customers, and V2 offers grounded in policy chunks.

In [1]:
# 1. Setup and imports
import os, sys, json, time, warnings
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ["RAG_PROMPT_VERSION"] = "V2"
pd.set_option("display.max_colwidth", 120, "display.width", 140)

EVIDENCE = ROOT / "docs" / "lab3-evidence"
FEATURES = json.loads((ROOT / "model" / "model_metadata.json").read_text())["feature_columns"]
print("Repo root:", ROOT.name, "| features:", len(FEATURES), "| prompt version:", os.environ["RAG_PROMPT_VERSION"])

Repo root: cs401r-caden02 | features: 11 | prompt version: V2


In [2]:
# 2. Evaluation evidence
metrics = json.loads((EVIDENCE / "evaluation_metrics.json").read_text())
display(Markdown("**evaluation_metrics.json**"))
display(pd.Series(metrics, name="value").to_frame())

slices = pd.read_csv(EVIDENCE / "slice-evaluation.csv")
display(Markdown("**slice-evaluation.csv** (`too_small=True` means the slice is too small for a reliable AUC)"))
display(slices)

**evaluation_metrics.json**

,value
auc_roc,0.782
precision_top10,0.71
recall_top10,0.3227
auc_vs_baseline,0.0587
baseline_auc,0.7232
lift_ci_low,0.0408
lift_ci_high,0.0776
positive_rate_val,0.22
n_val_samples,3000
eval_timestamp,2026-10-17T17:22:53.390131


**slice-evaluation.csv** (`too_small=True` means the slice is too small for a reliable AUC)

,tier,n,positives,auc,recall,too_small
0,Bronze,1071,361,0.749924,0.457064,False
1,Gold,483,52,0.750156,0.115385,False
2,Platinum,307,21,0.887446,0.095238,True
3,Silver,1139,226,0.720153,0.176991,False


In [3]:
# 3. Load model, score the test split, top-10% cutoff
# Features come from the Feature Store offline store (same query and split as training:
# 70/30, stratified, random_state=42, rows ordered by customer_id).
sm = boto3.client("sagemaker", region_name="us-east-1")
athena = boto3.client("athena", region_name="us-east-1")
account = boto3.client("sts").get_caller_identity()["Account"]  # used only to build the bucket name; never printed

fg = sm.describe_feature_group(FeatureGroupName="northstar-dev-customer-features")
table = fg["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]
database = fg["OfflineStoreConfig"]["DataCatalogConfig"]["Database"]
out = f"s3://northstar-dev-data-{account}/artifacts/athena-results/"

sql = f"""
WITH ranked AS (
  SELECT customer_id, {", ".join(FEATURES)}, churn_label, loyalty_tier,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY event_time DESC, write_time DESC) AS rn
  FROM "{table}" WHERE NOT is_deleted)
SELECT customer_id, {", ".join(FEATURES)}, churn_label, loyalty_tier FROM ranked WHERE rn = 1 ORDER BY customer_id"""
qid = athena.start_query_execution(QueryString=sql, QueryExecutionContext={"Database": database},
                                   ResultConfiguration={"OutputLocation": out})["QueryExecutionId"]
while True:
    st = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
    if st in ("SUCCEEDED", "FAILED", "CANCELLED"): break
    time.sleep(2)
assert st == "SUCCEEDED", f"Athena query {st}"
rows = []
for page in athena.get_paginator("get_query_results").paginate(QueryExecutionId=qid):
    rows += [[c.get("VarCharValue") for c in r["Data"]] for r in page["ResultSet"]["Rows"]]
df = pd.DataFrame(rows[1:], columns=rows[0])
for c in FEATURES + ["churn_label"]:
    df[c] = pd.to_numeric(df[c])
df = df[df["churn_label"].notna()].reset_index(drop=True)

X = df[FEATURES].fillna(-1).values
y = df["churn_label"].astype(int).values
idx = np.arange(len(df))
_, idx_test, _, y_test = train_test_split(idx, y, stratify=y, test_size=0.30, random_state=42)
test = df.iloc[idx_test].copy()

booster = xgb.Booster()
booster.load_model(str(ROOT / "model" / "model.xgb"))
test["score"] = booster.predict(xgb.DMatrix(test[FEATURES].fillna(-1).values, feature_names=FEATURES))

cutoff = np.quantile(test["score"], 0.90)
test["flagged"] = test["score"] >= cutoff
top = test[test["flagged"]]
print(f"Test rows: {len(test):,} | AUC: {roc_auc_score(test['churn_label'], test['score']):.4f} (evidence: {metrics['auc_roc']})")
print(f"Top-10% cutoff (score >= {cutoff:.4f}): {len(top)} flagged customers")
print(f"Precision in flagged: {top['churn_label'].mean():.3f} (evidence: {metrics['precision_top10']}) | "
      f"recall: {top['churn_label'].sum() / test['churn_label'].sum():.3f} (evidence: {metrics['recall_top10']})")
display(top["loyalty_tier"].value_counts().rename("flagged").to_frame())

Test rows: 3,000 | AUC: 0.7820 (evidence: 0.782)
Top-10% cutoff (score >= 0.6065): 300 flagged customers
Precision in flagged: 0.710 (evidence: 0.71) | recall: 0.323 (evidence: 0.3227)


/var/folders/w_/llwsf3dx0w70z39_9twqylzc0000gn/T/ipykernel_59568/1826287315.py:41: UserWarning: [11:54:55] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1489: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  booster.load_model(str(ROOT / "model" / "model.xgb"))


,flagged
loyalty_tier,
Bronze,225
Silver,66
Gold,6
Platinum,3


In [4]:
# 4. Three flagged customers from different tiers -> retrieved chunks + V2 offer
from rag.offer_pipeline import rag_offer_fn, last_retrieved_chunk_ids, prompt_version
assert prompt_version() == "V2"

raw = pd.read_csv(ROOT / "northstar-raw-sample.csv", usecols=["customer_id", "purchase_date", "product_category"])
raw = raw[raw["purchase_date"] <= "2026-04-01"]

def top_categories(cid, n=3):
    return raw.loc[raw["customer_id"] == cid, "product_category"].value_counts().head(n).index.tolist()

picks = []
for tier in ["Bronze", "Silver", "Gold", "Platinum"]:
    cand = top[top["loyalty_tier"] == tier].sort_values("score", ascending=False)
    if len(cand) and len(picks) < 3:
        picks.append(cand.iloc[0])

results = []
for r in picks:
    ctx = {
        "loyalty_tier": r["loyalty_tier"],
        "total_lifetime_value": round(float(r["total_lifetime_value"]), 2),
        "top_categories": top_categories(r["customer_id"]) or ["Apparel"],
        "churn_probability": round(float(r["score"]), 3),
        "days_since_last_purchase": int(r["days_since_last_purchase"]),
        "avg_order_value": round(float(r["avg_order_value"]), 2),
    }
    q = f"Generate a personalized retention offer for this at-risk {ctx['loyalty_tier']} member."
    out_ = rag_offer_fn(ctx, q)
    results.append((r["customer_id"], ctx, last_retrieved_chunk_ids(), out_["answer"]))

for cid, ctx, chunk_ids, answer in results:
    display(Markdown(f"### {cid} ({ctx['loyalty_tier']})"))
    display(pd.Series(ctx, name="profile").to_frame())
    print("Retrieved chunk IDs:")
    for c in chunk_ids:
        print("  -", c)
    display(Markdown(f"**V2 offer:** {answer}"))

### CUST-10006857 (Bronze)

,profile
loyalty_tier,Bronze
total_lifetime_value,158.6
top_categories,[Grocery]
churn_probability,0.965
days_since_last_purchase,305
avg_order_value,79.3


Retrieved chunk IDs:
  - loyalty-program-terms.md#4-tier-benefits-0
  - loyalty-program-terms.md#5-1-retention-offers-0
  - customer-faq.md#loyalty-0
  - loyalty-program-terms.md#5-promotional-offers-0
  - loyalty-program-terms.md#3-points-0


**V2 offer:** Earn 1.5× points on your next Grocery purchase to accelerate rewards faster and get free standard shipping when you spend $75 or more. As a Bronze member, you'll build your balance toward meaningful redemptions while enjoying the shipping benefit that makes shopping with us easier.

### CUST-10004252 (Silver)

...

**V2 offer:** We'd like to welcome you back with a special offer on Beauty products—as a Silver member, you'll earn 1.25x points on your purchase, plus enjoy free standard shipping on orders of $50 or more. This offer is valid for 45 days from today and can be combined with points redemption to maximize your rewards.

### CUST-10009715 (Gold)

,profile
loyalty_tier,Gold
total_lifetime_value,4244.73
top_categories,"[Apparel, Grocery, Outdoor]"
churn_probability,0.958
days_since_last_purchase,258
avg_order_value,385.88


Retrieved chunk IDs:
  - loyalty-program-terms.md#5-1-retention-offers-0
  - loyalty-program-terms.md#4-tier-benefits-0
  - customer-faq.md#loyalty-0
  - loyalty-program-terms.md#2-3-tier-protection-0
  - loyalty-program-terms.md#5-promotional-offers-0


**V2 offer:** We'd like to offer you **free return shipping on your next Apparel purchase**—a benefit exclusive to Gold members like you. As a Gold member, you also earn 1.5x points on every order and enjoy free standard shipping on any purchase, so your next trip to Apparel is fully supported. This offer is valid for 45 days from today.

In [5]:
# 5. V1 vs V2 RAG evaluation (mean per metric)
rag = pd.read_csv(EVIDENCE / "rag_eval.csv")
cols = ["faithfulness", "answer_relevancy", "context_recall"]
summary = rag.groupby("prompt_version")[cols].mean().T[["V1", "V2"]]
summary["delta (V2-V1)"] = summary["V2"] - summary["V1"]
display(Markdown(f"{rag.groupby('prompt_version').size().to_dict()} judged responses per version across runs; judge: {rag['judge_model'].iloc[0]}"))
display(summary.round(4))

{'V1': 12, 'V2': 12} judged responses per version across runs; judge: us.anthropic.claude-haiku-4-5-20251001-v1:0

prompt_version,V1,V2,delta (V2-V1)
faithfulness,0.4748,0.5685,0.0937
answer_relevancy,0.3373,0.4156,0.0783
context_recall,0.5583,0.5583,0.0000


## Limitations
- **Small RAG eval:** 4 test cases x 3 runs per prompt version (12 rows each). The V1 to V2 deltas are directional, not statistically established; context_recall is identical because retrieval did not change between versions.
- **LLM-as-judge:** faithfulness and relevancy are scored by a model (Haiku 4.5), so scores carry judge noise and bias.
- **Weak recall on high tiers:** top-10% recall is about 0.32 overall. Per-tier recall is low for Gold (about 0.12) and Platinum (about 0.10), and the Platinum slice is flagged `too_small`, so its AUC should not be trusted.
- **Tier definition mismatch:** the feature store tier is derived from lifetime value, while policy tiers use trailing 12-month spend, so an offer's tier claim can be wrong for some customers.
- **Demo profile data:** top categories in this notebook come from the raw transaction sample (up to the feature cutoff), and the offers are single generations at temperature 0.2, not a statistical sample.
- **Offers fail policy audits often:** in `docs/lab3-evidence/track-b-examples.md` I audited one V2 offer for each of 42 model-flagged customers against the full policy docs. Only **10 of 42 audited clean**, and 6 of those 10 carry a tier caveat (the customer is within 20% of the Bronze/Silver threshold), so just 4 are clean outright (3 Silver, 1 Bronze). No Gold or Platinum offer was clean (0 of 9). The most frequent failures were points "never expire" (12 of 42), a points multiplier from the wrong tier or an invented bonus (7), universal wording such as "every dollar" or "all orders" (7), and overstated returns or return shipping (5). Prompt rules alone do not prevent these.
- **Offers are not approved:** generated text has not passed a policy or human review and should not be sent to customers as is.